In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import StratifiedKFold

In [ ]:
dataset_name = "exploration"
post_impute = True # Check results of feature engineering or missing value patterns?

In [ ]:
train_df = pd.read_csv(f"/mnt/nextcloud/data_v11/{dataset_name}/train.csv")
test_df = pd.read_csv(f"/mnt/nextcloud/data_v11/{dataset_name}/test.csv")

In [ ]:
# check if there are columns with missing values and show missing percentage
missing = dict()
for col in train_df.columns:
    if any(train_df[col].isnull()):
        missing[col] = sum(train_df[col].isnull()) / len(train_df)
missing_df = pd.DataFrame(
    {"columns": missing.keys(), "percent_missing": missing.values()}
).sort_values("percent_missing", ascending=False)
missing_df

In [ ]:
# check if any columns are not encoded properly (only SPM_ID can show up here)
if post_impute: 
    train_df.select_dtypes("object")

## Post Impute

In [ ]:
# read version without imputing for imputation eval
if post_impute:
    pre_impute_train = pd.read_csv(f"/mnt/nextcloud/data_v11/dev/exploration/train.csv")
    pre_impute_test = pd.read_csv(f"/mnt/nextcloud/data_v11/dev/exploration/test.csv")                                                                                                                            

In [ ]:
# drop SPM_ID for imputation eval
#if post_impute:
#    pre_impute_train = pre_impute_train.drop(columns=["SPM_ID"])
#    pre_impute_test = pre_impute_test.drop(columns=["SPM_ID"])
#    train_df = train_df.drop(columns=["SPM_ID"])
#    test_df = test_df.drop(columns=["SPM_ID"])

In [ ]:
# get test scores for imputed data
if post_impute:
    param_grid = {
        "max_depth": [3, 7],
        "min_samples_leaf": [10, 30],
        "learning_rate": [0.01, 0.05, 0.1],
    }
    hgbc_impute = HistGradientBoostingClassifier(random_state=42)

    grid_search = RandomizedSearchCV(hgbc_impute, param_grid, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring='matthews_corrcoef', n_iter=5, n_jobs=-1)
    grid_search.fit(train_df.drop(columns=["Transplantationsstatus"]), train_df["Transplantationsstatus"])

    print("Best cross-validation score:", grid_search.best_score_)

    best_hgbc = grid_search.best_estimator_

    test_score = best_hgbc.score(test_df.drop(columns=["Transplantationsstatus"]), test_df["Transplantationsstatus"])
    print("Test score:", test_score)

In [ ]:
# get test scores for non-imputed data
if post_impute:
    hgbc = HistGradientBoostingClassifier(random_state=42)
    grid_search = RandomizedSearchCV(hgbc, param_grid, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring='matthews_corrcoef', n_iter=5, n_jobs=-1)
    grid_search.fit(pre_impute_train.drop(columns=["Transplantationsstatus"]), pre_impute_train["Transplantationsstatus"])

    print("Best cv score:", grid_search.best_score_)

    best_hgbc = grid_search.best_estimator_

    test_score = best_hgbc.score(pre_impute_test.drop(columns=["Transplantationsstatus"]), pre_impute_test["Transplantationsstatus"])
    print("Test score:", test_score)

In [ ]:
train_df.shape

## Pre-Impute

In [ ]:
# check if there are patterns in missing values
if not post_impute:
    plt.figure(figsize=(12, 100))
    cmap = ListedColormap(["darkgrey", "black"])
    sns.heatmap(train_df[missing.keys()].isnull(), cbar=False, cmap=cmap, linewidths=0.5)
    plt.yticks([], minor=False)
    plt.show()

In [ ]:
# check categorical columns
if not post_impute:
    cat_cols = train_df.select_dtypes(include="object").columns
    categrorical_df = pd.DataFrame({"name": cat_cols})
    categrorical_df["values"] = [train_df[n].unique() for n in categrorical_df.name]
    categrorical_df["missing_ratio"] = [sum(train_df[n].isnull()) / len(train_df) for n in categrorical_df.name]
    categrorical_df

In [ ]:
# check int columns
if not post_impute:
    int_cols = train_df.select_dtypes(include="int64").columns
    int_df = pd.DataFrame({"name": int_cols})
    int_df["values"] = [train_df[n].unique() for n in int_df.name]
    int_df["missing_ratio"] = [sum(train_df[n].isnull()) / len(train_df) for n in int_df.name]
    int_df

In [ ]:
# check float columns
if not post_impute:
    float_cols = train_df.select_dtypes(include="float64").columns
    float_df = pd.DataFrame({"name": float_cols})
    float_df["values"] = [train_df[n].unique() for n in float_df.name]
    float_df["missing_ratio"] = [sum(train_df[n].isnull()) / len(train_df) for n in float_df.name]
    float_df[float_df.missing_ratio > 0].sort_values("missing_ratio")

In [ ]:
# check boolean columns
if not post_impute:
    bool_cols = train_df.select_dtypes(include="bool").columns
    bool_df = pd.DataFrame({"name": bool_cols})
    bool_df["values"] = [train_df[n].unique() for n in bool_df.name]
    bool_df["missing_ratio"] = [sum(train_df[n].isnull()) / len(train_df) for n in bool_df.name]
    bool_df

In [ ]:
# check if all variables are included
if not post_impute:
    len(train_df.columns) - len(categrorical_df) - len(int_df) - len(float_df) - len(bool_df)